# Diffusion Trajectories versus OT Rays in Two Dimensions

This notebook generates `fig:generative-diffusion-versus-ot-2d`.  The same central noise particles and three-cluster target samples are connected either by straight OT displacement rays or by longer curved diffusion-style trajectories.  Each trajectory, including its initial disk, is colored by the final target cluster.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb
import ot

NAME = "generative-diffusion-versus-ot-2d"
out = figure_dir(NAME)
rng = np.random.default_rng(43)

The paths are synthetic and lightweight.  The curved panel emphasizes the qualitative difference between multi-step sampling paths and the straight displacement selected by the quadratic OT map.

In [2]:
centers = np.array([[-1.08, -0.38], [1.08, -0.36], [0.00, 1.08]])
counts = [12, 12, 12]
y = np.vstack([rng.multivariate_normal(c, [[0.018, 0.0], [0.0, 0.022]], n) for c, n in zip(centers, counts)])
cluster = np.repeat(np.arange(3), counts)
x = rng.normal(loc=[0.0, 0.0], scale=[0.34, 0.32], size=(len(y), 2))
a = np.ones(len(x))/len(x)
P = ot.emd(a, a, ot.dist(x, y))
match = P.argmax(axis=1)
ym = y[match]
cm = cluster[match]
all_points = np.vstack([x, y])
xlim, ylim = padded_limits(all_points, pad=0.14)
cluster_colors = [RED, VIOLET, BLUE]

In [3]:
def curved_path(p0, p1, cidx, steps=68):
    ts = np.linspace(0, 1, steps)
    d = p1 - p0
    perp = np.array([-d[1], d[0]]) / (np.linalg.norm(d) + 1e-12)
    radial = p1 / (np.linalg.norm(p1) + 1e-12)
    ctrl1 = p0 + 0.24 * radial + 0.22 * ((-1)**cidx) * perp
    ctrl2 = p1 - 0.30 * radial + 0.14 * ((-1)**(cidx+1)) * perp
    t = ts[:, None]
    return (1-t)**3*p0 + 3*(1-t)**2*t*ctrl1 + 3*(1-t)*t**2*ctrl2 + t**3*p1

def draw(paths, filename):
    fig, ax = plt.subplots(figsize=(2.45, 2.25))
    for i, pts in enumerate(paths):
        base = np.array(to_rgb(cluster_colors[int(cm[i])]))
        segs = np.stack([pts[:-1], pts[1:]], axis=1)
        cols = [(*base, 0.34) for _ in range(len(segs))]
        ax.add_collection(LineCollection(segs, colors=cols, linewidths=0.55, zorder=1))
    for k, col in enumerate(cluster_colors):
        idx0 = cm == k
        idy = cluster == k
        ax.scatter(x[idx0,0], x[idx0,1], s=DIRAC_MARKER_SIZE * 0.45, marker="o", color=col, edgecolor="none", linewidth=0, alpha=0.46, zorder=2)
        ax.scatter(y[idy,0], y[idy,1], s=DIRAC_MARKER_SIZE * 0.57, marker="o", color=col, edgecolor="none", linewidth=0, zorder=4)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.055)
    plt.close(fig)

ot_paths = [np.stack([x[i], ym[i]]) for i in range(len(x))]
diff_paths = [curved_path(x[i], ym[i], int(cm[i])) for i in range(len(x))]
draw(diff_paths, "diffusion-curves.pdf")
draw(ot_paths, "ot-rays.pdf")